In [1]:
from sage.all import *

class Berlekamp:
    """
    Sonlu cisimler üzerinde Berlekamp Çarpanlara Ayırma Algoritmasının
    adım adım izlenebilir (verbose) implementasyonu.

    Berlekamp algoritması, GF(q) üzerinde tanımlı kareçarpansız (square-free)
    bir f(x) polinomunu indirgenemez çarpanlarına ayırır. Temel fikir,
    Frobenius endomorfizmasının x^q ≡ x (mod p) özelliğinden yararlanarak
    bir Q matrisi inşa etmek ve bu matrisin çekirdeğindeki (nullspace)
    vektörlerle GCD hesabı yaparak çarpanları elde etmektir.

    Algoritmanın teorik dayanağı: F_q[x]/(f) bölüm halkasında,
    v(x)^q = v(x) denklemini sağlayan v(x) polinomlarının kümesi
    (Berlekamp cebiri), f'nin indirgenemez çarpan sayısı kadar
    boyuta sahip bir F_q-vektör uzayı oluşturur.

    Referans: Tez, Bölüm 4 — Tek Değişkenli Polinomların Çarpanlara Ayrılması
    """

    def __init__(self, field):
        """
        Sınıfı verilen sonlu cisim üzerinde başlatır.

        Parametreler
        ------------
        field : SageMath sonlu cismi (GF(q))
            Polinomların katsayılarının ait olduğu sonlu cisim.
            q, bir asal sayının kuvveti olmalıdır (q = p^m).
        """
        self.F = field
        self.q = field.order()                 # Cismin eleman sayısı q = p^m
        self.R = PolynomialRing(self.F, 'x')    # F_q[x] tek değişkenli polinom halkası
        self.x = self.R.gen()                   # Halkanın üreteci x

        print("\n" + "="*70)
        print(f"BERLEKAMP ALGORİTMASI ")
        print(f"Cisim: GF({self.q})")
        print("="*70)

    # ----------------------------------------------------------------
    # BÖLÜM 1: KARE-ÇARPANSIZLIK KONTROLÜ
    # ----------------------------------------------------------------
    def check_square_free(self, f):
        """
        Girdi polinomunun kare çarpansız (square-free) olup olmadığını denetler.

        Berlekamp algoritması yalnızca kare çarpansız polinomlar üzerinde
        doğru sonuç üretir; katlı kökler varsa Berlekamp cebirinin
        boyutu ile gerçek çarpan sayısı örtüşmez. Bu nedenle algoritma
        çalıştırılmadan önce bu ön koşulun sağlanması zorunludur.

        Kontrol iki aşamalıdır:
          (i)  f'(x) = 0 mu? -> f(x) = g(x)^p biçiminde olabilir
               (karakteristik p'de x^(kp) teriminin türevi sıfırdır).
          (ii) EBOB(f, f') = 1 mi? -> Değilse f ile f' ortak kökü
               paylaşıyordur, bu da f'nin katlı kökü olduğu anlamına gelir.

        Parametreler
        ------------
        f : F_q[x] polinomu
            Kare çarpansızlık durumu kontrol edilecek polinom.

        Döndürür
        --------
        bool
            f kare çarpansız ise True; sıfır polinom, sabit polinom veya
            katlı köklü ise False.
        """
        print(f"\n[ADIM 1] Kare-Çarpansızlık (Square-Free) Kontrolü...")

        # Sıfır polinom tanımsızdır; Berlekamp algoritması için anlamsızdır
        if f == 0:
            print("   -> HATA: Sıfır polinom için Berlekamp algoritması uygulanamaz.")
            return False

        # Derece 0 (sabit) polinomların çarpanlara ayrılacak bir yapısı yoktur
        if f.degree() <= 0:
            print("   -> Sabit polinom. Ayrıştırılacak bir polinom yok.")
            return False

        # Türev hesabı: klasik türev kuralları F_q[x] üzerinde de geçerlidir
        df = f.derivative()
        print(f"   -> Türev f'(x) hesaplandı: {df}")

        # Karakteristik p'de, x^(kp) biçimindeki her terimin türevi
        # (kp) * x^(kp-1) = 0 olur (çünkü kp ≡ 0 mod p).
        # Dolayısıyla f'(x) = 0 ise f(x) yalnızca p'nin katı olan
        # üslere sahip terimlerden oluşur, yani f(x) = g(x)^p'dir.
        if df == 0:
            print("   -> Türev 0! Polinom f(x) = g(x)^p formundadır.")
            print("   -> Bu nedenle girdi polinomu square-free değildir.")
            print("   -> Berlekamp algoritmasından önce Square-Free Factorization uygulanmalıdır.")
            return False

        # EBOB(f, f') hesaplanır: f ile f'nin ortak kökleri, f'nin
        # katlı köklerine karşılık gelir (bir kök f'de k kere tekrar
        # ediyorsa, f'de (k-1) kere tekrar eder).
        d = gcd(f, df)
        print(f"   -> d(x) = EBOB(f, f') = {d}")

        if d == 1:
            # Ortak kök yok -> tüm kökler basit (katlılık 1) -> kare çarpansız
            print("   -> EBOB(f, f') = 1. Polinom Square-Free.")
            print("   -> Berlekamp algoritması bu polinom üzerinde çalıştırılacaktır.")
            return True
        else:
            # Ortak kök var -> en az bir kök katlı -> kare çarpansız değil
            print("   -> EBOB(f, f') != 1. Polinom square-free değildir.")
            print("   -> Katlı çarpanlar bulunmaktadır.")
            print("   -> Berlekamp algoritmasından önce Square-Free Factorization uygulanmalıdır.")
            return False

    # ----------------------------------------------------------------
    # BÖLÜM 2: BERLEKAMP ÇEKİRDEK ALGORİTMASI
    # ----------------------------------------------------------------
    def berlekamp_core_solver(self, f):
        """
        Kare çarpansız olduğu doğrulanmış f(x) polinomu üzerinde Berlekamp
        algoritmasının çekirdek adımlarını uygular.

        Algoritma üç alt adımdan oluşur:

          2.1 Q Matrisinin İnşası
              Q, n×n boyutunda bir matristir (n = deg(f)).
              i. satırı, x^(i*q) mod f polinomunun katsayı vektörüdür.
              Bu matris, F_q[x]/(f) bölüm halkası üzerindeki Frobenius
              endomorfizmasının (a -> a^q) matris temsilidir.

          2.2 Çekirdek (Nullspace) Hesabı
              (Q - I) matrisinin sağ çekirdeği hesaplanır. Bu çekirdek,
              v(x)^q ≡ v(x) (mod f) denklemini sağlayan v(x)
              polinomlarının oluşturduğu Berlekamp cebirine karşılık
              gelir. Berlekamp'ın temel teoremine göre bu cebirin
              boyutu, f'nin farklı indirgenemez çarpan sayısına eşittir.

          2.3 Çarpanlara Ayırma (GCD Taraması)
              Çekirdekteki her v(x) vektörü ve her s ∈ F_q için
              gcd(u(x), v(x) - s) hesaplanır. Teorik garanti: eğer h(x),
              f'nin bir indirgenemez çarpanıysa, v(x) mod h(x) daima
              F_q'nun sabit bir elemanına eşittir; bu nedenle doğru s
              değeri seçildiğinde h(x), gcd hesabıyla ayrıştırılabilir.

        Parametreler
        ------------
        f : F_q[x] polinomu
            Kare çarpansız olduğu önceden doğrulanmış polinom.

        Döndürür
        --------
        list of F_q[x]
            f'nin indirgenemez çarpanlarının listesi (katlılıksız,
            her biri bir kez).
        """
        print(f"\n   >>> BERLEKAMP ANALİZİ BAŞLIYOR: {f} (Derece {f.degree()})")

        # Polinomu monik hale getir (baş katsayı 1); orijinal baş
        # katsayı, algoritmanın sonunda ilk çarpana geri çarpılacaktır.
        leading_coeff = f.leading_coefficient()
        f = f / leading_coeff
        n = f.degree()

        # ---------------------------------------------------------
        # 2.1 Q Matrisini Oluşturma ve Polinomları Yazdırma
        # ---------------------------------------------------------
        print(f"\n   [2.1] Q Matrisi Oluşturuluyor ({n}x{n})...")
        print(f"   (Her satır x^(i*{self.q}) mod f polinomuna karşılık gelir)\n")

        Q = matrix(self.F, n, n)

        for i in range(n):
            # i. satır: x^(i*q) mod f işleminin sonucu.
            # pow(x, k, f) kullanımı, x^k'yı doğrudan hesaplamak yerine
            # tekrarlı kare alma (fast exponentiation) ile f'e göre
            # modüler indirgeme yaparak büyük derece patlamasını önler.
            h_poly = pow(self.x, i * self.q, f)
            coeffs = h_poly.list()

            # Katsayı listesi n'den kısa olabilir (yüksek dereceli
            # terimlerin katsayısı sıfırsa); Q matrisinin ilgili
            # sütunları varsayılan olarak 0 kalır (örtük padding).
            for j in range(len(coeffs)):
                Q[i, j] = coeffs[j]

            # Her satırın hangi polinoma ve hangi katsayı vektörüne
            # karşılık geldiğini izlenebilirlik amacıyla yazdır
            print(f"   Satır {i}: x^{i*self.q} mod f = {h_poly}")
            print(f"            Vektör: {coeffs}")
            print("-" * 50)

        print("\n   >>> OLUŞAN BERLEKAMP MATRİSİ (Q):")
        print(Q)

        # ---------------------------------------------------------
        # 2.2 Çekirdek (Nullspace) Hesabı
        # ---------------------------------------------------------
        print(f"\n   [2.2] Q - I Matrisinin Çekirdeği Hesaplanıyor...")

        # v(x)^q ≡ v(x) (mod f) denklemi, katsayı vektörleri düzeyinde
        # v * Q = v biçiminde ifade edilir; bu da v * (Q - I) = 0
        # homojen denklemine, yani (Q - I)'nın sol çekirdeğine karşılık
        # gelir. SageMath'te sol çekirdek doğrudan tanımlı olmadığından
        # devrik (transpose) alınarak sağ çekirdek (right_kernel) ile
        # eşdeğer sonuç elde edilir.
        M = Q - identity_matrix(self.F, n)
        kernel = M.transpose().right_kernel()
        basis = kernel.basis()

        print(f"      -> Çekirdek Boyutu: {len(basis)}")
        print("      -> Bu değer square-free polinomun indirgenemez çarpan sayısına karşılık gelir.")

        # Çekirdek boyutu 1 ise Berlekamp cebiri yalnızca sabit
        # polinomlardan (F_q'nun elemanlarından) oluşur; bu da f'nin
        # zaten indirgenemez olduğu anlamına gelir.
        if len(basis) == 1:
            print("      -> Boyut 1. Bu polinom zaten İNDİRGENEMEZ.")
            return [f * leading_coeff]

        print("      -> Taban Vektörleri:")
        for idx, vec in enumerate(basis):
            print(f"         v{idx}: {vec}")

        # ---------------------------------------------------------
        # 2.3 Ayrıştırma (GCD Döngüsü)
        # ---------------------------------------------------------
        print(f"\n   [2.3] Çarpanlara Ayırma (GCD Döngüsü)...")

        # factors listesi, o ana kadar bulunmuş (henüz tam ayrışmamış
        # olabilecek) parçaları tutar; başlangıçta tek eleman f'dir
        factors = [f]

        for k in range(len(basis)):
            vec = basis[k]
            v_poly = self.R(vec.list())

            # Sabit (derece <= 0) bir çekirdek vektörü herhangi bir
            # ayırt edici bilgi taşımaz (v(x) - s zaten sabittir);
            # bu tür vektörler ayrıştırma için kullanışsızdır, atlanır.
            if v_poly.degree() <= 0:
                continue

            print(f"\n      -> Vektör v{k} ile tarama yapılıyor: {v_poly}")
            new_factor_list = []

            for u in factors:
                # Derece 1 (veya daha küçük) parçalar zaten
                # indirgenemezdir; daha fazla bölünmeye ihtiyaç yoktur
                if u.degree() <= 1:
                    new_factor_list.append(u)
                    continue

                current_pieces = [u]

                # GF(q)'nun her elemanı s için gcd(piece, v(x) - s)
                # hesaplanır. u'nun içindeki her indirgenemez çarpan h
                # için v(x) mod h sabit bir s0 değerine eşit olduğundan,
                # s = s0 seçildiğinde h, gcd sonucunda ortaya çıkar.
                for s in self.F:
                    if not current_pieces:
                        break

                    next_pieces = []

                    for piece in current_pieces:
                        # Derece 1 parçalar indirgenemez kabul edilir,
                        # daha fazla bölme denenmez
                        if piece.degree() <= 1:
                            next_pieces.append(piece)
                            continue

                        g = gcd(piece, v_poly - s)

                        # Gerçek bir ayrışma ancak g, piece'in ne
                        # kendisine (tam bölünme) ne de sabite (1)
                        # eşit olmadığı durumda gerçekleşir
                        if 1 < g.degree() < piece.degree():
                            print(f"         [AYRIŞTI!] s={s} için EBOB bulundu: {g}")
                            print(f"                    Diğer parça: {piece // g}")

                            next_pieces.append(g)
                            next_pieces.append(piece // g)
                        else:
                            # Ayrışma olmadı; parça değişmeden bir
                            # sonraki s denemesine aktarılır
                            next_pieces.append(piece)

                    current_pieces = next_pieces

                new_factor_list.extend(current_pieces)

            factors = new_factor_list

            # Bulunan çarpan sayısı çekirdek boyutuna (yani beklenen
            # indirgenemez çarpan sayısına) ulaştıysa döngü erken
            # sonlandırılabilir; kalan vektörler gereksizdir
            if len(factors) == len(basis):
                print("      -> Hedeflenen çarpan sayısına ulaşıldı.")
                break

        # Monikleştirme adımında ayrılan baş katsayı, sonuçların
        # çarpımı orijinal f'ye eşit olsun diye ilk çarpana geri eklenir
        final_factors = []
        for i, fac in enumerate(factors):
            if i == 0:
                final_factors.append(fac * leading_coeff)
            else:
                final_factors.append(fac)

        return final_factors

    # ----------------------------------------------------------------
    # ANA ÇALIŞTIRICI
    # ----------------------------------------------------------------
    def run(self, f):
        """
        Girdi polinomu üzerinde uçtan uca Berlekamp iş akışını yürütür.

        İş akışı:
          1. check_square_free() ile ön koşul kontrolü yapılır.
             Kare çarpansız değilse algoritma durdurulur (Berlekamp yalnızca
             kare çarpansız polinomlar için tanımlıdır).
          2. berlekamp_core_solver() ile indirgenemez çarpanlar bulunur.
          3. Bulunan çarpanların çarpımı hesaplanarak orijinal f ile
             karşılaştırılır (doğruluk sağlaması).

        Parametreler
        ------------
        f : F_q[x] polinomu
            Çarpanlarına ayrılacak polinom.

        Döndürür
        --------
        list of F_q[x]
            f kare çarpansız ise indirgenemez çarpanların listesi;
            kare çarpansız değilse boş liste.
        """
        print(f"\n>>> İŞLEM BAŞLATILIYOR <<<")
        print(f"Girdi Polinomu: {f}")
        print("-" * 40)

        # 1. Ön koşul: girdi polinomu kare çarpansız mı?
        is_square_free = self.check_square_free(f)

        if not is_square_free:
            print("\n" + "="*70)
            print("İŞLEM DURDURULDU")
            print("="*70)
            print("Bu girdi square-free olmadığı için önce Square-Free Factorization uygulanmalıdır.")
            print("="*70)
            return []

        # 2. Kare çarpansızlık doğrulandı; Berlekamp çekirdek algoritmasını çalıştır
        all_factors = self.berlekamp_core_solver(f)

        print("\n" + "="*70)
        print("SONUÇLAR:")
        check_poly = self.R(1)

        for i, fac in enumerate(all_factors):
            print(f"   Çarpan {i+1}: {fac}")
            check_poly *= fac

        print("-" * 70)
        print(f"Çarpanların çarpımı: {check_poly}")
        print(f"Girdi polinomu     : {f}")

        # 3. Doğruluk sağlaması: bulunan çarpanların çarpımı, orijinal
        # girdi polinomuna (baş katsayı dahil) tam olarak eşit olmalıdır
        if check_poly == f:
            print("SAĞLAMA: BAŞARILI (Çarpım Doğru)")
        else:
            print(f"SAĞLAMA: HATA! (Hesaplanan: {check_poly})")

        print("="*70)

        return all_factors




In [2]:
# ============================================================
# SENARYO 1: Manuel cisim ve manuel polinom
# ============================================================
# GF(5) üzerinde, üç bilinen indirgenemez çarpanın (derece 1, 2 ve 3)
# çarpımından elle bir polinom oluşturulur. Bu senaryo, beklenen
# sonucun önceden bilindiği kontrollü bir test ortamı sağlar.

q = 5
F = GF(q)

solver = Berlekamp(F)
R = solver.R
x = solver.x

# Manuel polinom girişi: (derece 1) * (derece 2) * (derece 3)
f = (x + 1) * (x**2 + x + 2) * (x**3 + 2*x + 1)

print("\nManuel girilen polinom:")
print(f"f(x) = {f}")
print(f"deg(f) = {f.degree()}")

solver.run(f)



BERLEKAMP ALGORİTMASI 
Cisim: GF(5)

Manuel girilen polinom:
f(x) = x^6 + 2*x^5 + 2*x^3 + 3*x^2 + 2*x + 2
deg(f) = 6

>>> İŞLEM BAŞLATILIYOR <<<
Girdi Polinomu: x^6 + 2*x^5 + 2*x^3 + 3*x^2 + 2*x + 2
----------------------------------------

[ADIM 1] Kare-Çarpansızlık (Square-Free) Kontrolü...
   -> Türev f'(x) hesaplandı: x^5 + x^2 + x + 2
   -> d(x) = EBOB(f, f') = 1
   -> EBOB(f, f') = 1. Polinom Square-Free.
   -> Berlekamp algoritması bu polinom üzerinde çalıştırılacaktır.

   >>> BERLEKAMP ANALİZİ BAŞLIYOR: x^6 + 2*x^5 + 2*x^3 + 3*x^2 + 2*x + 2 (Derece 6)

   [2.1] Q Matrisi Oluşturuluyor (6x6)...
   (Her satır x^(i*5) mod f polinomuna karşılık gelir)

   Satır 0: x^0 mod f = 1
            Vektör: [1]
--------------------------------------------------
   Satır 1: x^5 mod f = x^5
            Vektör: [0, 0, 0, 0, 0, 1]
--------------------------------------------------
   Satır 2: x^10 mod f = 4*x^5 + 4*x^3 + 4*x^2 + 3*x + 3
            Vektör: [3, 3, 4, 4, 0, 4]
------------------

[x^3 + 2*x + 1, x^2 + x + 2, x + 1]

In [3]:

# ============================================================
# SENARYO 2: Derece listesine göre rastgele çarpanlı polinom
# ============================================================
# GF(5) üzerinde, 'degrees' listesinde belirtilen derecelerde
# rastgele seçilmiş indirgenemez çarpanların çarpımından bir polinom
# üretilir. Bu senaryo, algoritmayı farklı çarpan sayısı ve derece
# kombinasyonlarıyla (örn. tekrarlı dereceler) test etmeye olanak tanır.

q = 5
F = GF(q)

solver = Berlekamp(F)
R = solver.R
x = solver.x

# Hangi derecelerden indirgenemez çarpanlar üretileceğini burada belirle.
# Örnek:
# degrees = [3, 2, 1]      -> 3., 2. ve 1. dereceden birer indirgenemez çarpan
# degrees = [2, 2, 2, 3]   -> üç tane 2. derece, bir tane 3. derece çarpan
degrees = [3, 2, 1, 2]

f = R(1)
factors = []

for d in degrees:
    g = R.irreducible_element(d, algorithm="random")

    # Aynı indirgenemez çarpanın birden fazla seçilmesi f'yi katlı
    # köklü hale getirir (square-free koşulunu bozar); bu nedenle
    # zaten seçilmiş bir çarpan tekrar çıkarsa yeniden örneklenir.
    while g in factors:
        g = R.irreducible_element(d, algorithm="random")

    factors.append(g)
    f *= g

print("\nSeçilen indirgenemez çarpanlar:")
for g in factors:
    print(f"   {g}")

print("\nOluşturulan polinom:")
print(f"f(x) = {f}")
print(f"deg(f) = {f.degree()}")

solver.run(f)



BERLEKAMP ALGORİTMASI 
Cisim: GF(5)

Seçilen indirgenemez çarpanlar:
   x^3 + 3*x + 3
   x^2 + 4*x + 1
   x + 4
   x^2 + 3

Oluşturulan polinom:
f(x) = x^8 + 3*x^7 + 3*x^6 + x^3 + 2*x^2 + 4*x + 1
deg(f) = 8

>>> İŞLEM BAŞLATILIYOR <<<
Girdi Polinomu: x^8 + 3*x^7 + 3*x^6 + x^3 + 2*x^2 + 4*x + 1
----------------------------------------

[ADIM 1] Kare-Çarpansızlık (Square-Free) Kontrolü...
   -> Türev f'(x) hesaplandı: 3*x^7 + x^6 + 3*x^5 + 3*x^2 + 4*x + 4
   -> d(x) = EBOB(f, f') = 1
   -> EBOB(f, f') = 1. Polinom Square-Free.
   -> Berlekamp algoritması bu polinom üzerinde çalıştırılacaktır.

   >>> BERLEKAMP ANALİZİ BAŞLIYOR: x^8 + 3*x^7 + 3*x^6 + x^3 + 2*x^2 + 4*x + 1 (Derece 8)

   [2.1] Q Matrisi Oluşturuluyor (8x8)...
   (Her satır x^(i*5) mod f polinomuna karşılık gelir)

   Satır 0: x^0 mod f = 1
            Vektör: [1]
--------------------------------------------------
   Satır 1: x^5 mod f = x^5
            Vektör: [0, 0, 0, 0, 0, 1]
-------------------------------------------

[x^2 + 4*x + 1, x + 4, x^2 + 3, x^3 + 3*x + 3]